# Biomedical Concept Extraction + Normalization

This notebook extracts phenotype and disease terms from freeform clinical text and normalizes each term to a CURIE using the SRI Name Resolution API, scored with BioBERT and SapBERT cosine similarity.

## Workflow

1. Load your tabulated clinical text dataset.
2. Run an LLM over each row to extract phenotype and disease terms as structured JSON.
3. For each extracted term, query the SRI Name Resolution API for candidate CURIEs.
4. Score each candidate using BioBERT (local) and SapBERT (API) cosine similarity.
   - BioBERT compares the original extracted term against each candidate label **and** synonyms, taking the max cosine.
   - SapBERT returns its own embedding-based scores by CURIE.
5. The best match is the candidate with the highest cosine score across both models.
6. Export a review-ready Excel file with term-level and source-level sheets.

## Data Use Guardrails

Only use data that is:
- Synthetic, publicly shareable, or de-identified
- Approved for this environment

Do **not** upload PHI, PII, credentials, restricted datasets, or sensitive internal data.

## Required Packages

```
pip install openai pandas openpyxl python-dotenv requests transformers torch numpy
```

In [ ]:
# ============================================================
# Notebook Setup + Controls
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import os
import sys
import time
import hashlib

import pandas as pd
import numpy as np
import requests
from requests.adapters import HTTPAdapter, Retry
from IPython.display import display
from dotenv import load_dotenv

# ------------------------------------------------------------
# 1. Row limit — keep small for first runs
# ------------------------------------------------------------
MAX_ROWS_TO_RUN = 5

# ------------------------------------------------------------
# 2. Dataset path
#    Update BYOD_DATA_PATH to point to your file.
#    Supported: .csv, .xlsx, .xls
# ------------------------------------------------------------
BYOD_DATA_PATH = Path("../data/your_dataset.xlsx")  # <-- UPDATE THIS

# ------------------------------------------------------------
# 3. Column names
#    TEXT_COLUMN: the column containing freeform clinical text
#    ROW_ID_COLUMN: a unique identifier per row (or set to None
#                   to auto-generate from row index)
# ------------------------------------------------------------
TEXT_COLUMN   = "patient_description"  # <-- UPDATE THIS
ROW_ID_COLUMN = "record_id"            # <-- UPDATE THIS (or set to None)

# ------------------------------------------------------------
# 4. API settings
# ------------------------------------------------------------
NR_API_URL   = "https://name-resolution-sri.renci.org/lookup"
NR_PREFIXES  = "HP|MONDO|UMLS|NCIT|OMIM|ORPHANET"  # ontology filters
NR_LIMIT     = 25

SAPBERT_URL   = "https://sap-qdrant.apps.renci.org/annotate/"
SAPBERT_COUNT = 25

# ------------------------------------------------------------
# 5. BioBERT model
# ------------------------------------------------------------
BIOBERT_MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"

# ------------------------------------------------------------
# 6. Output / cache directories
# ------------------------------------------------------------
OUTPUT_DIR = Path("outputs")
LOG_DIR    = Path("logs")
CACHE_DIR  = Path("cache") / "concept_extraction"

for d in [OUTPUT_DIR, LOG_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------
print("Notebook setup complete.")
print(f"Python version : {sys.version.split()[0]}")
print(f"Data path      : {BYOD_DATA_PATH}")
print(f"Text column    : {TEXT_COLUMN}")
print(f"Row ID column  : {ROW_ID_COLUMN}")
print(f"Max rows       : {MAX_ROWS_TO_RUN}")
print(f"Run timestamp  : {RUN_TIMESTAMP}")
print(f"Output folder  : {OUTPUT_DIR.resolve()}")

## Configure the AI Provider

This notebook uses an OpenAI-compatible client for term extraction.
The Name Resolution and SapBERT services are RENCI-hosted REST APIs that require no additional credentials.

Paste the workshop/sandbox API key below, or set `OPENAI_API_KEY` in a `.env` file.

In [ ]:
# ============================================================
# API Key Configuration
# ============================================================

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY", "PASTE_YOUR_API_KEY_HERE")

if not API_KEY or API_KEY.strip() == "":
    raise ValueError("API_KEY is blank. Please set OPENAI_API_KEY or paste it above.")

if API_KEY == "PASTE_YOUR_API_KEY_HERE":
    raise ValueError(
        "API_KEY is still the placeholder. "
        "Replace it with your actual key before running."
    )

print("API key configured successfully.")
print(f"Key preview: {API_KEY[:8]}...{API_KEY[-4:]}")

In [ ]:
# ============================================================
# Initialize AI Client + Smoke Test
# ============================================================

from openai import OpenAI

MODEL_NAME = "gpt-4.1-mini"  # update if your sandbox uses a different model

client = OpenAI(api_key=API_KEY, timeout=60.0, max_retries=2)

try:
    _test = client.responses.create(
        model=MODEL_NAME,
        instructions="Reply with exactly: API connection successful.",
        input="Connection test.",
        max_output_tokens=20,
    )
    print(f"AI client ready. Model: {MODEL_NAME}")
    print(f"Smoke test: {_test.output_text}")
except Exception as e:
    print(f"Connection test failed: {e}")


def call_ai_model(instructions, prompt, max_output_tokens=900, temperature=0, model_name=MODEL_NAME):
    """
    Send one prompt to the configured AI model.
    Returns (raw_output, success_bool, error_str).
    """
    try:
        resp = client.responses.create(
            model=model_name,
            instructions=instructions,
            input=prompt,
            temperature=temperature,
            max_output_tokens=max_output_tokens,
        )
        return getattr(resp, "output_text", "").strip(), True, ""
    except Exception as e:
        return "", False, str(e)


print("call_ai_model helper ready.")

## Load and Preview Your Dataset

Update `BYOD_DATA_PATH`, `TEXT_COLUMN`, and `ROW_ID_COLUMN` in the setup cell before running this step.

Review the column list and preview to confirm the right fields are loaded.

In [ ]:
# ============================================================
# Load Dataset
# ============================================================

if not BYOD_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {BYOD_DATA_PATH.resolve()}\n"
        "Update BYOD_DATA_PATH in the setup cell."
    )

file_suffix = BYOD_DATA_PATH.suffix.lower()

if file_suffix in [".xlsx", ".xls"]:
    excel_file = pd.ExcelFile(BYOD_DATA_PATH)
    print("Available sheets:", excel_file.sheet_names)
    SHEET_NAME = 0  # update if your data is on a different sheet
    df = pd.read_excel(BYOD_DATA_PATH, sheet_name=SHEET_NAME)
elif file_suffix == ".csv":
    df = pd.read_csv(BYOD_DATA_PATH)
else:
    raise ValueError("Unsupported file type. Use .csv, .xlsx, or .xls")

starting_shape = df.shape
df = df.dropna(how="all").dropna(axis=1, how="all")

print(f"\nLoaded: {starting_shape[0]} rows x {starting_shape[1]} columns")
print(f"Cleaned: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nColumn names:")
for col in df.columns:
    print(f"  - {col}")

print("\nFirst 5 rows:")
display(df.head())

missing = df.isna().sum().reset_index()
missing.columns = ["column", "missing_count"]
missing["pct_missing"] = (missing["missing_count"] / len(df) * 100).round(1)
print("\nMissingness:")
display(missing)

## Map Columns and Build Row Contexts

Each row context carries the row identifier and the clinical text that will be sent to the LLM.

Only the text column is passed to the model. The row ID stays outside the model call for merging later.

In [ ]:
# ============================================================
# Map Columns and Build Row Contexts
# ============================================================

if TEXT_COLUMN not in df.columns:
    raise ValueError(
        f"TEXT_COLUMN '{TEXT_COLUMN}' not found.\n"
        f"Available columns: {list(df.columns)}"
    )

if ROW_ID_COLUMN is not None and ROW_ID_COLUMN not in df.columns:
    print(f"WARNING: ROW_ID_COLUMN '{ROW_ID_COLUMN}' not found. Using row index as ID.")
    ROW_ID_COLUMN = None

df_work = df.copy()


def clean_cell_value(value):
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return str(value).strip()


prototype_df = df_work.head(MAX_ROWS_TO_RUN).copy()
row_contexts = []

for source_index, row in prototype_df.iterrows():
    row_id = (
        clean_cell_value(row[ROW_ID_COLUMN])
        if ROW_ID_COLUMN
        else str(source_index)
    )
    text = clean_cell_value(row[TEXT_COLUMN])
    row_contexts.append({
        "source_index": source_index,
        "row_id":       row_id,
        "text":         text,
    })

print(f"Prepared {len(row_contexts)} row contexts.")
print(f"Text column    : {TEXT_COLUMN}")
print(f"Row ID column  : {ROW_ID_COLUMN or '(row index)'}")
print("\nExample row context:")
print(json.dumps(row_contexts[0] if row_contexts else {}, indent=2, ensure_ascii=False))

## Load Extraction Task Definitions

The extraction task is defined in `docs/definitions_biomedical-concept-extraction.md`.

That file specifies what to extract, the allowed categories, confidence definitions, and the expected JSON output schema.
Keeping the specification in a separate file makes it easy to refine without rewriting the notebook.

**Important:** The allowed category and confidence lists in this cell must stay synchronized with the definitions file.

In [ ]:
# ============================================================
# Load Extraction Task Definitions
# ============================================================

DEFINITIONS_PATH_OPTIONS = [
    Path("docs/definitions_biomedical-concept-extraction.md"),
    Path("../docs/definitions_biomedical-concept-extraction.md"),
]

DEFINITIONS_PATH = next((p for p in DEFINITIONS_PATH_OPTIONS if p.exists()), None)

if DEFINITIONS_PATH is None:
    raise FileNotFoundError(
        "Could not find definitions_biomedical-concept-extraction.md.\n"
        "Expected at: docs/definitions_biomedical-concept-extraction.md"
    )

EXTRACTION_DEFINITIONS = DEFINITIONS_PATH.read_text(encoding="utf-8").strip()

# These lists must match the categories and confidence levels in the definitions file.
ALLOWED_CATEGORIES = [
    "phenotype",
    "disease",
    "syndrome",
    "symptom",
    "finding",
]

ALLOWED_CONFIDENCE = ["High", "Medium", "Low"]

print(f"Definitions loaded from: {DEFINITIONS_PATH}")
print(f"Definition length: {len(EXTRACTION_DEFINITIONS):,} characters")
print(f"\nAllowed categories : {ALLOWED_CATEGORIES}")
print(f"Allowed confidence : {ALLOWED_CONFIDENCE}")
print("\nDefinitions preview (first 600 chars):")
print(EXTRACTION_DEFINITIONS[:600])

## Build LLM Extraction Instructions

The extraction instructions combine the definitions file content with the allowed labels and a strict JSON output schema.

Preview the instructions and a sample prompt before running the extraction loop.

In [ ]:
# ============================================================
# Build LLM Extraction Instructions
# ============================================================

EXTRACTION_INSTRUCTIONS = f"""
You are assisting with a biomedical concept extraction task.

Your job is to extract potential phenotype and disease terms from clinical text.

Task specification:
{EXTRACTION_DEFINITIONS}

Allowed categories:
{json.dumps(ALLOWED_CATEGORIES, indent=2)}

Allowed confidence levels:
{json.dumps(ALLOWED_CONFIDENCE, indent=2)}

Return a single valid JSON object with exactly this structure:
{{
  "extracted_terms": [
    {{
      "term": "the extracted term, cleaned and specific",
      "category": "one of the allowed categories",
      "evidence_quote": "short excerpt from the text that supports this term",
      "confidence": "High, Medium, or Low"
    }}
  ],
  "extraction_notes": "optional note about ambiguity or anything worth flagging"
}}

Rules:
- Only extract terms grounded in the text. Do not invent or infer beyond what is stated.
- Prefer specific terms (e.g., 'bilateral sensorineural hearing loss' over 'hearing problem').
- If no phenotype or disease terms are found, return an empty extracted_terms list.
- Return valid JSON only. No markdown, no extra text before or after the JSON.
""".strip()


def build_extraction_prompt(text):
    return (
        "Extract all biomedical phenotype and disease terms from the following clinical text:\n\n"
        + text
    )


print("Extraction instructions ready.")
print(f"Instruction length: {len(EXTRACTION_INSTRUCTIONS):,} characters")

if row_contexts:
    print("\nExample prompt:")
    print("-" * 60)
    print(build_extraction_prompt(row_contexts[0]["text"])[:600])
    print("-" * 60)

## Test the LLM on One Row

Before running the full extraction loop, test the prompt on one row.

Confirm that:
- The model call succeeds
- The response parses as valid JSON
- The extracted terms use allowed categories and confidence levels
- The terms and evidence quotes look reasonable for your data

If the output does not look right, revise the definitions file or prompt before continuing.

In [ ]:
# ============================================================
# Parse + Validate Helpers
# ============================================================

def parse_model_json(response_text):
    """Parse JSON from model output, stripping markdown fences if present."""
    cleaned = response_text.strip()
    for fence in ["```json", "```"]:
        if cleaned.startswith(fence):
            cleaned = cleaned[len(fence):].strip()
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3].strip()
    return json.loads(cleaned)


def validate_extraction(parsed):
    """Return a list of validation error strings, or empty list if valid."""
    errors = []
    if "extracted_terms" not in parsed:
        errors.append("Missing required field: extracted_terms")
        return errors
    if not isinstance(parsed["extracted_terms"], list):
        errors.append("extracted_terms must be a list")
        return errors
    for i, t in enumerate(parsed["extracted_terms"]):
        for field in ["term", "category", "evidence_quote", "confidence"]:
            if field not in t:
                errors.append(f"Term[{i}]: missing field '{field}'")
        if t.get("category") not in ALLOWED_CATEGORIES:
            errors.append(f"Term[{i}]: invalid category '{t.get('category')}'")
        if t.get("confidence") not in ALLOWED_CONFIDENCE:
            errors.append(f"Term[{i}]: invalid confidence '{t.get('confidence')}'")
    return errors


# ============================================================
# Single-row test
# ============================================================

if row_contexts:
    _test_row = row_contexts[0]
    print(f"Testing row ID: {_test_row['row_id']}")

    _raw, _ok, _err = call_ai_model(
        instructions=EXTRACTION_INSTRUCTIONS,
        prompt=build_extraction_prompt(_test_row["text"]),
        max_output_tokens=900,
    )

    if not _ok:
        print(f"Model call failed: {_err}")
    else:
        try:
            _parsed = parse_model_json(_raw)
            _errors = validate_extraction(_parsed)
            print(f"Extracted {len(_parsed.get('extracted_terms', []))} terms")
            print("\nParsed output:")
            print(json.dumps(_parsed, indent=2, ensure_ascii=False))
            print("\nValidation errors:", _errors if _errors else "None")
        except Exception as e:
            print(f"Parse failed: {e}")
            print("Raw output:", _raw[:500])

## Run LLM Extraction Loop

Run the extraction across all prototype rows.

Each row produces a list of extracted terms with category, evidence quote, and confidence.
Results are stored in `extraction_results` and summarized in `extraction_df`.

In [ ]:
# ============================================================
# LLM Extraction Loop
# ============================================================

# Number of additional attempts after the first parse failure.
# Retries use a small temperature bump (0.3) and an explicit JSON reminder
# to get different output — temperature=0 retries are deterministic and pointless.
MAX_PARSE_RETRIES = 2


def extract_terms_from_row(row_ctx):
    base_prompt = build_extraction_prompt(row_ctx["text"])

    parsed    = {"extracted_terms": []}
    parse_ok  = False
    parse_err = ""
    api_ok    = False
    last_raw  = ""

    for attempt in range(1 + MAX_PARSE_RETRIES):
        if attempt == 0:
            prompt      = base_prompt
            temperature = 0
        else:
            prompt = (
                base_prompt
                + "\n\nIMPORTANT: Your previous response could not be parsed as JSON. "
                "Return ONLY a valid JSON object — no markdown fences, no explanatory text "
                "before or after the JSON."
            )
            temperature = 0.3
            print(f"    Row {row_ctx['row_id']}: parse failed, retry {attempt}/{MAX_PARSE_RETRIES}...")

        raw, api_ok, err = call_ai_model(
            instructions=EXTRACTION_INSTRUCTIONS,
            prompt=prompt,
            max_output_tokens=900,
            temperature=temperature,
        )
        last_raw = raw

        if not api_ok:
            parse_err = err
            continue  # try again if retries remain

        try:
            parsed    = parse_model_json(raw)
            parse_ok  = True
            parse_err = ""
            break  # success
        except Exception as e:
            parse_err = str(e)
            # will retry if attempts remain

    val_errors = validate_extraction(parsed) if parse_ok else ["Parsing failed"]

    return {
        "source_index":       row_ctx["source_index"],
        "row_id":             row_ctx["row_id"],
        "text":               row_ctx["text"],
        "extracted_terms":    parsed.get("extracted_terms", []),
        "extraction_notes":   parsed.get("extraction_notes", ""),
        "model_call_success": api_ok,
        "parse_success":      parse_ok,
        "parse_error":        parse_err,
        "validation_errors":  "; ".join(val_errors),
        "raw_model_output":   last_raw,
    }


extraction_results = []
print(f"Running extraction on {len(row_contexts)} rows...\n")

for i, row_ctx in enumerate(row_contexts, 1):
    print(f"Row {i}/{len(row_contexts)} | ID: {row_ctx['row_id']}")
    result = extract_terms_from_row(row_ctx)
    extraction_results.append(result)
    time.sleep(0.25)

extraction_df = pd.DataFrame(extraction_results)

total_terms    = sum(len(r["extracted_terms"]) for r in extraction_results)
failed_parses  = sum(1 for r in extraction_results if not r["parse_success"])

print(f"\nExtraction complete.")
print(f"Total terms extracted : {total_terms}")
print(f"Rows with parse errors: {failed_parses}")

if failed_parses > 0:
    print("\nRows that failed to parse (check raw_model_output for debugging):")
    display(
        extraction_df[~extraction_df["parse_success"]][
            ["row_id", "parse_error", "raw_model_output"]
        ].assign(raw_model_output=extraction_df["raw_model_output"].str[:200])
    )

display(
    extraction_df[["row_id", "parse_success", "validation_errors"]]
    .assign(term_count=extraction_df["extracted_terms"].apply(len))
)

## Flatten Extracted Terms

Convert the list-of-terms structure into one row per term.

Each row in `terms_df` is one extracted term tied back to its source row via `source_index`.
This is the unit that will be normalized in the next steps.

In [ ]:
# ============================================================
# Flatten Extracted Terms to One Row per Term
# ============================================================

flat_terms = []
for result in extraction_results:
    for term_dict in result["extracted_terms"]:
        flat_terms.append({
            "source_index":        result["source_index"],
            "row_id":              result["row_id"],
            "original_text":       result["text"],
            "term":                term_dict.get("term", "").strip(),
            "category":            term_dict.get("category", ""),
            "evidence_quote":      term_dict.get("evidence_quote", ""),
            "extraction_confidence": term_dict.get("confidence", ""),
        })

terms_df = pd.DataFrame(flat_terms)

# Drop blank terms
terms_df = terms_df[terms_df["term"].str.strip() != ""].reset_index(drop=True)

print(f"Total terms to normalize: {len(terms_df)}")
print(f"Unique terms: {terms_df['term'].nunique()}")

display(
    terms_df[["row_id", "term", "category", "extraction_confidence", "evidence_quote"]]
)

## Load BioBERT

BioBERT is a locally-run embedding model. It is used to compute cosine similarity between the extracted term and each Name Resolution candidate (label + all synonyms).

**First-time use**: the model weights (~440 MB) will be downloaded from HuggingFace and cached locally.

If `transformers` or `torch` are not installed, BioBERT scoring will be skipped and only SapBERT scores will be used.

```
pip install transformers torch
```

In [ ]:
# ============================================================
# Load BioBERT
# ============================================================

try:
    from transformers import AutoTokenizer, AutoModel
    import torch
    HAS_BIOBERT = True
    print("transformers + torch available.")
except ImportError:
    HAS_BIOBERT = False
    print("WARNING: transformers / torch not installed.")
    print("BioBERT scoring will be skipped. Install with: pip install transformers torch")


if HAS_BIOBERT:
    print(f"Loading BioBERT: {BIOBERT_MODEL_NAME}")
    print("(First run downloads ~440 MB of model weights — subsequent runs use the cache.)")
    biobert_tokenizer = AutoTokenizer.from_pretrained(BIOBERT_MODEL_NAME)
    biobert_model     = AutoModel.from_pretrained(BIOBERT_MODEL_NAME)
    biobert_model.eval()
    print("BioBERT loaded successfully.")


def _embed_strings(texts):
    """
    Return a (N, 768) numpy array of mean-pooled BioBERT embeddings.
    Processes in batches of 64 to stay within memory limits.
    Returns a zero matrix if BioBERT is unavailable.
    """
    if not HAS_BIOBERT or not texts:
        return np.zeros((len(texts), 768))

    all_vecs = []
    batch_size = 64
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        with torch.no_grad():
            enc  = biobert_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=64,
            )
            out  = biobert_model(**enc)
            vecs = out.last_hidden_state.mean(dim=1).numpy()
        all_vecs.append(vecs)
    return np.vstack(all_vecs)


print("_embed_strings helper ready.")

## Name Resolution API

The [SRI Name Resolution API](https://name-resolution-sri.renci.org/docs) maps a text string to candidate CURIEs from ontologies including HP, MONDO, UMLS, NCIT, OMIM, and ORPHANET.

Each candidate includes a label, a relevance score, a list of synonyms, and biolink type annotations.

Results are cached to disk so repeated runs do not re-query the API.

In [ ]:
# ============================================================
# Name Resolution API Helper
# ============================================================

def _make_session():
    s = requests.Session()
    retries = Retry(
        total=3,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    return s


NR_SESSION  = _make_session()
NR_CACHE_DIR = CACHE_DIR / "name_resolution"
NR_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def query_name_resolver(term, limit=NR_LIMIT, prefixes=NR_PREFIXES):
    """
    Query the SRI Name Resolution API for candidate CURIEs.
    Results are disk-cached by (term, limit, prefixes) hash.

    Returns a list of dicts:
      [{curie, label, score, types, synonyms}, ...]
    """
    payload = {"string": term, "limit": limit, "only_prefixes": prefixes}
    key     = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    cache_f = NR_CACHE_DIR / f"{key}.json"

    if cache_f.exists():
        return json.loads(cache_f.read_text())

    try:
        time.sleep(0.2)
        resp = NR_SESSION.get(NR_API_URL, params=payload, timeout=15)
        resp.raise_for_status()
        result = resp.json()
    except Exception as e:
        print(f"  NR error for '{term}': {e}")
        result = []

    cache_f.write_text(json.dumps(result))
    return result


print("Name Resolver helper ready.")
print("\nRunning API smoke test (hypotonia)...")
_nr_test = query_name_resolver("hypotonia", limit=3)
print(f"Results: {len(_nr_test)} candidates")
if _nr_test:
    print(json.dumps(_nr_test[0], indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# Expanded NR Query: Qualifier Stripping + Multi-Query
# ============================================================
#
# Name Resolution does string-based matching. A term like
# "absence of visual tracking" will surface many "absence of X"
# candidates because the qualifier dominates the query string.
# By also querying the core concept ("visual tracking"), we give
# the candidate pool a much better chance of containing semantically
# correct matches.

import re

_QUALIFIER_PATTERNS = [
    r'^absence of\s+',
    r'^absent\s+',
    r'^loss of\s+',
    r'^lack of\s+',
    r'^abnormality of\s+',
    r'^abnormal\s+',
    r'^decreased\s+',
    r'^reduced\s+',
    r'^increased\s+',
    r'^elevated\s+',
    r'^impaired\s+',
    r'^bilateral\s+',
    r'^unilateral\s+',
    r'^congenital\s+',
    r'^acquired\s+',
    r'^delayed\s+',
    r'^progressive\s+',
    r'^generalized\s+',
    r'^focal\s+',
    r'^chronic\s+',
    r'^severe\s+',
    r'^mild\s+',
    r'^moderate\s+',
]


def strip_qualifier(term):
    """
    Strip a single leading qualifier phrase and return the core concept string.
    Returns None if no known qualifier is found or if stripping produces an empty result.
    """
    term_lc = term.lower().strip()
    for pattern in _QUALIFIER_PATTERNS:
        stripped = re.sub(pattern, '', term_lc, count=1, flags=re.IGNORECASE).strip()
        if stripped and stripped != term_lc:
            return stripped
    return None


def get_nr_queries(term):
    """
    Return a deduplicated list of NR query strings for a term.
    Always includes the original term. Also includes the stripped core concept
    if a leading qualifier was detected.
    """
    queries = [term]
    core    = strip_qualifier(term)
    if core and core not in queries:
        queries.append(core)
    return queries


def query_name_resolver_expanded(term, limit=NR_LIMIT, prefixes=NR_PREFIXES):
    """
    Query NR with the original term and any qualifier-stripped variants.
    Merges all results by CURIE, keeping the highest NR score per candidate.
    Returns the merged list sorted by NR score descending.
    """
    queries = get_nr_queries(term)
    merged  = {}  # curie -> candidate dict

    for q in queries:
        for cand in query_name_resolver(q, limit=limit, prefixes=prefixes):
            curie = cand.get("curie", "")
            if curie not in merged or cand.get("score", 0) > merged[curie].get("score", 0):
                merged[curie] = cand

    return sorted(merged.values(), key=lambda c: c.get("score", 0), reverse=True)


# Demo
print("Expanded NR query helper ready.")
_demo = "absence of visual tracking"
print(f"\nDemo: '{_demo}'")
print(f"  NR queries: {get_nr_queries(_demo)}")

## SapBERT API

SapBERT is a biomedical entity-linking model hosted at RENCI. It returns the top-N concept matches for a text string, with cosine similarity scores in the 0–1 range.

SapBERT scores are looked up by CURIE against the Name Resolution candidates, and also used directly if SapBERT returns a top result not found in the NR candidates.

Results are cached to disk.

In [ ]:
# ============================================================
# SapBERT API Helper
# ============================================================

SAP_SESSION  = _make_session()
SAP_CACHE_DIR = CACHE_DIR / "sapbert"
SAP_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def query_sapbert(term, count=SAPBERT_COUNT):
    """
    Query the SapBERT embedding service for top-N concept matches.
    Results are disk-cached by (term, count) hash.

    Returns a list sorted by score descending:
      [{score, category, name, curie}, ...]
    """
    key     = hashlib.sha256(f"sapbert:{term}:{count}".encode()).hexdigest()
    cache_f = SAP_CACHE_DIR / f"{key}.json"

    if cache_f.exists():
        return json.loads(cache_f.read_text())

    try:
        time.sleep(0.2)
        resp = SAP_SESSION.post(
            SAPBERT_URL,
            json={"text": term, "model_name": "sapbert", "count": count},
            timeout=15,
        )
        resp.raise_for_status()
        result = resp.json()
        if not isinstance(result, list):
            result = []
    except Exception as e:
        print(f"  SapBERT error for '{term}': {e}")
        result = []

    cache_f.write_text(json.dumps(result))
    return result


print("SapBERT helper ready.")
print("\nRunning API smoke test (hypotonia)...")
_sap_test = query_sapbert("hypotonia", count=3)
print(f"Results: {len(_sap_test)} candidates")
if _sap_test:
    print(json.dumps(_sap_test[0], indent=2, ensure_ascii=False))

## Cosine Similarity and Candidate Scoring

For each extracted term:

1. **BioBERT scoring**: compare the original term embedding against each NR candidate's label embedding AND each synonym embedding. The candidate's BioBERT score is the **maximum** cosine across the label and all synonyms. This captures cases where the term best matches a synonym rather than the preferred label. All comparisons use **lowercase** versions of the strings so capitalization does not affect cosine similarity.

2. **SapBERT scoring**: look up each NR candidate's CURIE in the SapBERT results. If SapBERT returns a top result not present in the NR candidates, add it as an additional candidate.

3. **Best match selection**: the candidate with the **highest single cosine score from either model** wins — not an average or combination. A perfect BioBERT score stands on its own regardless of its SapBERT score, and vice versa. When two candidates share the exact same top score, the tiebreaker is the sum of both model scores (favoring the candidate that scored well in both models).

In [ ]:
# ============================================================
# Cosine Similarity + Candidate Scoring
# ============================================================

def _cosine(a, b):
    """Cosine similarity between two numpy vectors. Returns nan if either is None."""
    if a is None or b is None:
        return float("nan")
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


def score_candidates(original_term, nr_candidates, sapbert_results, embed_map):
    """
    Score NR candidates using BioBERT (label + synonyms, take max) and SapBERT (by CURIE).
    Returns (best_match_dict, other_candidates_list).

    embed_map keys are lowercase strings. All lookups lowercase before accessing.

    BioBERT strategy:
      - For each candidate: cosine(original_term_vec, label_vec)
                             cosine(original_term_vec, synonym_vec) for each synonym
      - All strings are lowercased before embedding lookup.
      - Candidate BioBERT score = max cosine across label and all synonyms.
      - biobert_matched_string records the original-case string that gave the max.

    SapBERT strategy:
      - Look up each candidate CURIE in the SapBERT result list.
      - Top-1 SapBERT result is added as a candidate even if not in NR list.

    Best match selection:
      - Primary sort key: max(biobert_cosine, sapbert_cosine) — highest single score wins.
      - Tiebreaker: biobert_cosine + sapbert_cosine — candidate scoring well in both wins ties.
    """
    q_vec        = embed_map.get(original_term.lower())
    sap_by_curie = {
        r.get("curie", ""): round(float(r.get("score", float("nan"))), 4)
        for r in sapbert_results
    }

    scored = []

    for cand in nr_candidates:
        curie    = cand.get("curie", "")
        label    = cand.get("label", "") or cand.get("name", "")
        synonyms = [s for s in cand.get("synonyms", []) if s and str(s).strip()]
        nr_score = cand.get("score")

        # BioBERT: compare original term vs label + every synonym (all lowercased)
        comparison_strings = [label] + synonyms
        bio_pairs = []
        for s in comparison_strings:
            s_lc = s.lower() if s else ""
            if s_lc and s_lc in embed_map:
                cos = _cosine(q_vec, embed_map[s_lc])
                if not np.isnan(cos):
                    bio_pairs.append((s, round(cos, 4)))

        if bio_pairs:
            best_bio_str, best_bio_cos = max(bio_pairs, key=lambda x: x[1])
        else:
            best_bio_str, best_bio_cos = "", None

        sap_cos = sap_by_curie.get(curie)  # None if not in SapBERT results

        scored.append({
            "curie":                  curie,
            "label":                  label,
            "nr_score":               nr_score,
            "biobert_cosine":         best_bio_cos,
            "biobert_matched_string": best_bio_str,
            "sapbert_cosine":         sap_cos,
            "synonyms":               synonyms,
        })

    # Add pure SapBERT top-1 if not already in NR candidates
    nr_curies = {c["curie"] for c in scored}
    for sap_hit in sapbert_results[:1]:  # only top-1 to avoid noise
        s_curie = sap_hit.get("curie", "")
        if s_curie and s_curie not in nr_curies:
            scored.append({
                "curie":                  s_curie,
                "label":                  sap_hit.get("name", ""),
                "nr_score":               None,
                "biobert_cosine":         None,
                "biobert_matched_string": "",
                "sapbert_cosine":         round(float(sap_hit.get("score", 0.0)), 4),
                "synonyms":               [],
            })

    if not scored:
        return None, []

    def _sort_key(c):
        bio = c.get("biobert_cosine") or 0.0
        sap = c.get("sapbert_cosine") or 0.0
        return (max(bio, sap), bio + sap)  # primary: highest single score; tiebreaker: sum

    best = max(scored, key=_sort_key)

    bio_w = best.get("biobert_cosine") or 0.0
    sap_w = best.get("sapbert_cosine") or 0.0
    if bio_w == 0.0 and sap_w == 0.0:
        winning_model = "nr_fallback"
    elif bio_w > sap_w:
        winning_model = "biobert"
    elif sap_w > bio_w:
        winning_model = "sapbert"
    else:
        winning_model = "both_tied"  # same score from both models

    best_match = {
        "label":                  best["label"],
        "curie":                  best["curie"],
        "biobert_cosine":         best["biobert_cosine"],
        "sapbert_cosine":         best["sapbert_cosine"],
        "winning_model":          winning_model,
        "biobert_matched_string": best.get("biobert_matched_string", ""),
    }

    others = sorted(
        [
            {
                "label":          c["label"],
                "curie":          c["curie"],
                "nr_score":       c["nr_score"],
                "biobert_cosine": c["biobert_cosine"],
                "sapbert_cosine": c["sapbert_cosine"],
            }
            for c in scored
            if c["curie"] != best["curie"]
        ],
        key=_sort_key,
        reverse=True,
    )

    return best_match, others


print("Cosine similarity and scoring helpers ready.")

## Test Normalization on One Term

Before running the full normalization loop, test the complete pipeline on the first extracted term.

Confirm that:
- The Name Resolution API returns candidates with labels and synonyms
- SapBERT returns scored results
- BioBERT embeddings are computed without error
- The best match and other candidates look reasonable

If there are API connection errors, check your network access to `name-resolution-sri.renci.org` and `sap-qdrant.apps.renci.org`.

In [ ]:
# ============================================================
# Test Normalization on One Term
# ============================================================

if len(terms_df) > 0:
    _test_term_row = terms_df.iloc[0]
    _test_term     = _test_term_row["term"]

    print(f"Testing normalization for: '{_test_term}'")
    print(f"Category: {_test_term_row['category']}")
    print(f"Evidence: {_test_term_row['evidence_quote']}")

    # Query APIs
    _nr_results  = query_name_resolver(_test_term)
    _sap_results = query_sapbert(_test_term)
    print(f"\nNR candidates : {len(_nr_results)}")
    print(f"SapBERT results: {len(_sap_results)}")

    # Collect strings, then lowercase before embedding
    _strings_to_embed = set([_test_term])
    for c in _nr_results:
        lbl = c.get("label", "")
        if lbl:
            _strings_to_embed.add(lbl)
        for s in c.get("synonyms", []):
            if s:
                _strings_to_embed.add(s)

    # Lowercase all strings — embed_map keyed by lowercase
    _str_list = list({s.lower() for s in _strings_to_embed})
    print(f"\nStrings to embed, lowercased (term + labels + synonyms): {len(_str_list)}")
    print("Generating BioBERT embeddings...")

    _vecs      = _embed_strings(_str_list)
    _embed_map = {s: _vecs[i] for i, s in enumerate(_str_list)}
    print("Embeddings complete.")

    # Score
    _best_match, _others = score_candidates(_test_term, _nr_results, _sap_results, _embed_map)

    print("\nBest match:")
    print(json.dumps(_best_match, indent=2, ensure_ascii=False))

    print(f"\nOther candidates (top {min(5, len(_others))})")
    print(json.dumps(_others[:5], indent=2, ensure_ascii=False))

else:
    print("No terms extracted yet. Run the extraction loop first.")

## Run Normalization Loop

Run the full normalization pipeline over all extracted terms.

For efficiency, BioBERT embeddings are computed in one batch across all unique strings (extracted terms + all candidate labels + all candidate synonyms) before per-term scoring.

API results are cached so the loop can be interrupted and resumed.

In [ ]:
# ============================================================
# Normalization Loop
# ============================================================

def normalize_all_terms(terms_df):
    """
    Normalize all extracted terms.

    Steps:
      1. Query NR API for each unique term using expanded multi-query (qualifier stripping).
      2. Query SapBERT API for each unique term.
      3. Collect all strings (terms + labels + synonyms), lowercase them all,
         and embed in one BioBERT batch. embed_map is keyed by lowercase strings.
         embed_map is exposed as a global so the LLM review step can extend it.
      4. Score each term's candidates with BioBERT (label + synonyms max, lowercased)
         and SapBERT. Best match = highest single cosine score; tiebreaker = sum.
      5. Return one result row per term.
    """
    global embed_map  # expose for LLM review second-pass embedding

    if terms_df.empty:
        print("No terms to normalize.")
        return []

    unique_terms = [t for t in terms_df["term"].dropna().unique() if t]
    print(f"Normalizing {len(unique_terms)} unique terms ({len(terms_df)} total rows)...")

    # --------------------------------------------------------
    # Step 1: Query NR API (expanded: original + qualifier-stripped variants)
    # --------------------------------------------------------
    print("\nStep 1/4: Name Resolution API (expanded multi-query)...")
    nr_map = {}
    for i, term in enumerate(unique_terms, 1):
        print(f"  [{i}/{len(unique_terms)}] {term}")
        nr_map[term] = query_name_resolver_expanded(term)

    # --------------------------------------------------------
    # Step 2: Query SapBERT API
    # --------------------------------------------------------
    print("\nStep 2/4: SapBERT API...")
    sap_map = {}
    for i, term in enumerate(unique_terms, 1):
        print(f"  [{i}/{len(unique_terms)}] {term}")
        sap_map[term] = query_sapbert(term)

    # --------------------------------------------------------
    # Step 3: Collect all strings, lowercase, and batch-embed
    # --------------------------------------------------------
    print("\nStep 3/4: Collecting strings for BioBERT batch embedding...")
    all_strings_set = set(unique_terms)
    for term in unique_terms:
        for cand in nr_map.get(term, []):
            lbl = cand.get("label", "")
            if lbl:
                all_strings_set.add(lbl)
            for syn in cand.get("synonyms", []):
                if syn:
                    all_strings_set.add(syn)

    # Lowercase everything — embed_map is keyed by lowercase strings
    all_strings = list({s.lower() for s in all_strings_set})
    print(f"  {len(all_strings)} unique strings, lowercased (terms + labels + synonyms)")
    print("  Generating BioBERT embeddings...")
    vecs      = _embed_strings(all_strings)
    embed_map = {s: vecs[i] for i, s in enumerate(all_strings)}
    print("  Embeddings complete.")

    # --------------------------------------------------------
    # Step 4: Score each unique term
    # --------------------------------------------------------
    print("\nStep 4/4: Scoring candidates...")
    normalized_by_term = {}
    for term in unique_terms:
        best_match, others = score_candidates(
            term,
            nr_map.get(term, []),
            sap_map.get(term, []),
            embed_map,
        )
        normalized_by_term[term] = {
            "best_match":       best_match,
            "other_candidates": others,
        }

    # --------------------------------------------------------
    # Build output rows (one per extracted term)
    # --------------------------------------------------------
    output_rows = []
    for _, row in terms_df.iterrows():
        term   = row["term"]
        result = normalized_by_term.get(term, {"best_match": None, "other_candidates": []})
        bm     = result["best_match"]

        output_rows.append({
            "source_index":              row["source_index"],
            "row_id":                    row["row_id"],
            "term":                      term,
            "category":                  row["category"],
            "evidence_quote":            row["evidence_quote"],
            "extraction_confidence":     row["extraction_confidence"],
            "best_match_label":          bm["label"]          if bm else None,
            "best_match_curie":          bm["curie"]          if bm else None,
            "best_match_biobert_cosine": bm["biobert_cosine"] if bm else None,
            "best_match_sapbert_cosine": bm["sapbert_cosine"] if bm else None,
            "winning_model":             bm["winning_model"]  if bm else None,
            "biobert_matched_string":    bm.get("biobert_matched_string", "") if bm else "",
            "other_candidates_json":     json.dumps(result["other_candidates"][:10], ensure_ascii=False),
            "normalization_output_json": json.dumps(
                {
                    "original_term":         term,
                    "category":              row["category"],
                    "evidence_quote":        row["evidence_quote"],
                    "extraction_confidence": row["extraction_confidence"],
                    "best_match":            bm,
                    "other_candidates":      result["other_candidates"][:10],
                },
                ensure_ascii=False,
            ),
        })

    return output_rows


normalized_rows = normalize_all_terms(terms_df)
normalized_df   = pd.DataFrame(normalized_rows)

print(f"\nNormalization complete. {len(normalized_df)} term rows.")

display(
    normalized_df[[
        "row_id", "term",
        "best_match_label", "best_match_curie",
        "winning_model",
        "best_match_biobert_cosine", "best_match_sapbert_cosine",
    ]]
)

## LLM Semantic Review for Low-Confidence Matches

The BioBERT and SapBERT cosine scores measure embedding proximity — but they can be fooled by structural patterns.

**The problem:** A term like `absence of visual tracking` shares the `absence of [X]` pattern with many unrelated HPO terms (e.g., `HP:0001849 absence of toes`). The qualifier `absence of` dominates the string, producing high cosine scores despite the terms being semantically unrelated.

**The solution — three layers:**

1. **Qualifier-stripped multi-query NR** (applied in the normalization step above): each term is queried against both the full string *and* the qualifier-stripped core concept (e.g., also querying `visual tracking`). This expands the candidate pool.

2. **LLM semantic review**: for any term whose best cosine score falls below `LLM_REVIEW_THRESHOLD` (default: 0.99), the top candidates are sent to the LLM for semantic judgment. The LLM assesses whether any candidate accurately represents the same clinical concept.

3. **Second-pass NR with suggested query**: if the LLM finds no good match and suggests a better search string, the pipeline re-queries NR + SapBERT with that string, re-scores against the original term, and runs the LLM review again on the new candidates.

### LLM Review Output Columns

| Column | Description |
|---|---|
| `llm_review_quality` | `good`, `partial`, `none`, `skip` (score ≥ threshold), `parse_failed`, `review_failed` |
| `llm_review_curie` | CURIE recommended by the LLM (may differ from best cosine match) |
| `llm_review_label` | Label for the LLM-recommended CURIE |
| `llm_review_reasoning` | LLM's 1-2 sentence explanation |
| `llm_suggested_query` | Alternative search term used for second-pass NR |

Rows with `llm_review_quality` of `none`, `partial`, `parse_failed`, or `review_failed` are additionally flagged `needs_human_review = True`.

In [ ]:
# ============================================================
# LLM Semantic Review: Threshold, Instructions, and Function
# ============================================================

# Any best-match cosine below this threshold triggers LLM semantic review.
LLM_REVIEW_THRESHOLD = 0.99

LLM_REVIEW_INSTRUCTIONS = """
You are a biomedical curation assistant. Your job is to judge whether a candidate CURIE
accurately represents the same clinical concept as an extracted term.

You will receive:
  - original_term: the extracted clinical term (e.g., "absence of visual tracking")
  - category: the biomedical category (phenotype, disease, syndrome, symptom, finding)
  - candidates: a list of up to 8 candidate CURIEs with their labels and cosine scores

Your task:
1. Identify the candidate that BEST matches the original term.
2. Assess how well that candidate represents the same clinical concept.
3. If no candidate is a good match, suggest a better search query that would find the correct concept.

Judgment criteria:
- "good"    : the best candidate accurately represents the same clinical concept as the original term.
              The label or synonyms clearly describe the same finding, disease, or phenotype.
- "partial" : the best candidate is related but not an exact match. It may be broader, narrower,
              or describe a related but distinct concept.
- "none"    : no candidate in the list accurately represents the original term.
              The candidates are semantically unrelated despite superficial string similarity.

Return a single valid JSON object:
{
  "recommended_curie"      : "the CURIE of the best matching candidate, or null if none is good",
  "recommended_label"      : "the label of the recommended candidate, or null",
  "semantic_match_quality" : "good", "partial", or "none",
  "reasoning"              : "1-2 sentences explaining your judgment",
  "suggested_query"        : "if quality is none or partial, suggest an alternative search string that would better find the correct concept, otherwise null"
}

Rules:
- Return valid JSON only. No markdown, no extra text before or after the JSON.
- Do not recommend a candidate simply because it has a high cosine score — judge the actual meaning.
- When suggesting a query, strip structural qualifiers (like 'absence of', 'bilateral', 'impaired')
  and use the core clinical concept (e.g., 'visual tracking' not 'absence of visual tracking').
""".strip()


def llm_review_match(original_term, category, candidates, top_n=8):
    """
    Ask the LLM to semantically review the top NR candidates for a term.

    Returns a dict:
      {
        "recommended_curie"      : str or None,
        "recommended_label"      : str or None,
        "semantic_match_quality" : "good" | "partial" | "none" | "parse_failed" | "review_failed",
        "reasoning"              : str,
        "suggested_query"        : str or None,
      }
    """
    failed = {
        "recommended_curie":      None,
        "recommended_label":      None,
        "semantic_match_quality": "review_failed",
        "reasoning":              "",
        "suggested_query":        None,
    }

    if not candidates:
        return {**failed, "reasoning": "No candidates to review.", "semantic_match_quality": "none"}

    prompt = json.dumps({
        "original_term": original_term,
        "category":      category,
        "candidates":    candidates[:top_n],
    }, indent=2, ensure_ascii=False)

    raw, ok, err = call_ai_model(
        instructions=LLM_REVIEW_INSTRUCTIONS,
        prompt=prompt,
        max_output_tokens=400,
        temperature=0,
    )

    if not ok:
        return {**failed, "reasoning": f"LLM call failed: {err}"}

    try:
        parsed = parse_model_json(raw)
        return {
            "recommended_curie":      parsed.get("recommended_curie"),
            "recommended_label":      parsed.get("recommended_label"),
            "semantic_match_quality": parsed.get("semantic_match_quality", "parse_failed"),
            "reasoning":              parsed.get("reasoning", ""),
            "suggested_query":        parsed.get("suggested_query"),
        }
    except Exception as e:
        return {
            **failed,
            "semantic_match_quality": "parse_failed",
            "reasoning": f"JSON parse error: {e}. Raw: {raw[:200]}",
        }


print("LLM review instructions and llm_review_match helper ready.")
print(f"Review threshold: cosine < {LLM_REVIEW_THRESHOLD}")

In [ ]:
# ============================================================
# Run LLM Semantic Review
# ============================================================

def run_llm_review(df):
    """
    For each term with best cosine < LLM_REVIEW_THRESHOLD, run an LLM semantic
    review of the top NR candidates. If the LLM suggests a better query term,
    run a second-pass NR + scoring with that term.

    Adds columns:
      llm_review_quality  : "good" | "partial" | "none" | "skip" | "parse_failed" | "review_failed"
      llm_review_curie    : recommended CURIE (or None if quality == "none"/"skip")
      llm_review_label    : recommended label (or None)
      llm_review_reasoning: LLM's reasoning text
      llm_suggested_query : LLM-suggested alternative search term (or None)
    """
    df = df.copy()

    for col in ["llm_review_quality", "llm_review_curie", "llm_review_label",
                "llm_review_reasoning", "llm_suggested_query"]:
        if col not in df.columns:
            df[col] = None

    # Copy the global embed_map built during normalization; expand in-place for second-pass strings.
    local_embed_map = dict(embed_map)

    for idx, row in df.iterrows():
        best_bio  = row.get("best_match_biobert_cosine") or 0.0
        best_sap  = row.get("best_match_sapbert_cosine") or 0.0
        top_score = max(best_bio, best_sap)

        if top_score >= LLM_REVIEW_THRESHOLD:
            df.at[idx, "llm_review_quality"] = "skip"
            continue

        term      = row["term"]
        category  = row["category"]
        curie_now = row.get("best_match_curie")

        try:
            candidates = json.loads(row.get("other_candidates_json") or "[]")
        except Exception:
            candidates = []

        # Prepend the current best match so the LLM sees it too
        if curie_now:
            candidates = [{
                "label":          row.get("best_match_label"),
                "curie":          curie_now,
                "biobert_cosine": best_bio or None,
                "sapbert_cosine": best_sap or None,
            }] + candidates

        print(f"  LLM review: '{term}' (best cosine: {top_score:.4f})")
        review = llm_review_match(term, category, candidates, top_n=8)

        df.at[idx, "llm_review_quality"]  = review.get("semantic_match_quality")
        df.at[idx, "llm_review_curie"]     = review.get("recommended_curie")
        df.at[idx, "llm_review_label"]     = review.get("recommended_label")
        df.at[idx, "llm_review_reasoning"] = review.get("reasoning")
        df.at[idx, "llm_suggested_query"]  = review.get("suggested_query")

        # --------------------------------------------------------
        # Second-pass NR if LLM gave a suggested query
        # --------------------------------------------------------
        suggested = review.get("suggested_query") or ""
        if suggested and review.get("semantic_match_quality") in ("none", "partial"):
            print(f"    Second-pass NR → suggested query: '{suggested}'")

            sp_nr  = query_name_resolver_expanded(suggested)
            sp_sap = query_sapbert(suggested)

            # Embed any new strings not yet in local_embed_map
            new_strings = {suggested.lower()}
            for c in sp_nr:
                lbl = (c.get("label") or "").lower()
                if lbl and lbl not in local_embed_map:
                    new_strings.add(lbl)
                for syn in c.get("synonyms", []):
                    s_lc = (syn or "").lower()
                    if s_lc and s_lc not in local_embed_map:
                        new_strings.add(s_lc)

            if new_strings:
                new_list = list(new_strings)
                new_vecs = _embed_strings(new_list)
                for s, v in zip(new_list, new_vecs):
                    local_embed_map[s] = v

            # Score with the *original* extracted term as the query vector
            sp_best, sp_others = score_candidates(term, sp_nr, sp_sap, local_embed_map)

            if sp_best:
                sp_bio_cos = sp_best.get("biobert_cosine") or 0.0
                sp_sap_cos = sp_best.get("sapbert_cosine") or 0.0
                sp_score   = max(sp_bio_cos, sp_sap_cos)

                if sp_score > top_score:
                    sp_candidates = [{
                        "label":          sp_best["label"],
                        "curie":          sp_best["curie"],
                        "biobert_cosine": sp_bio_cos or None,
                        "sapbert_cosine": sp_sap_cos or None,
                    }] + sp_others[:7]

                    sp_review = llm_review_match(term, category, sp_candidates, top_n=8)

                    if sp_review.get("semantic_match_quality") in ("good", "partial"):
                        print(f"    Second-pass accepted: {sp_best['curie']} ({sp_best['label']})")
                        df.at[idx, "llm_review_quality"]  = sp_review.get("semantic_match_quality")
                        df.at[idx, "llm_review_curie"]     = sp_review.get("recommended_curie") or sp_best["curie"]
                        df.at[idx, "llm_review_label"]     = sp_review.get("recommended_label") or sp_best["label"]
                        df.at[idx, "llm_review_reasoning"] = (
                            f"[second-pass: '{suggested}'] " + (sp_review.get("reasoning") or "")
                        )
                        df.at[idx, "llm_suggested_query"]  = suggested

    return df


print("run_llm_review helper ready.")
print(f"\nRunning LLM semantic review (threshold: cosine < {LLM_REVIEW_THRESHOLD})...")
normalized_df = run_llm_review(normalized_df)

review_counts = normalized_df["llm_review_quality"].value_counts(dropna=False)
print("\nLLM review quality distribution:")
print(review_counts.to_string())

## Build Final Output per Source Row

Merge all term-level normalization results back to the source rows.

Each source row gets a `extracted_terms_json` column containing a JSON array of all extracted and normalized terms, each with:
- `original_term`
- `category`
- `evidence_quote`
- `extraction_confidence`
- `best_match` (label, CURIE, BioBERT cosine, SapBERT cosine, winning model, matched string)
- `other_candidates` (up to 10 alternatives with label, CURIE, nr_score, BioBERT cosine, SapBERT cosine)

In [ ]:
# ============================================================
# Merge Results Back to Source Rows
# ============================================================

final_rows = []

if not normalized_df.empty:
    for source_index, group in normalized_df.groupby("source_index"):
        source_row = df_work.iloc[source_index]
        row_id     = group["row_id"].iloc[0]
        text       = clean_cell_value(source_row[TEXT_COLUMN])

        terms_json = []
        for _, trow in group.iterrows():
            try:
                terms_json.append(json.loads(trow["normalization_output_json"]))
            except Exception:
                pass

        final_rows.append({
            "source_index":         source_index,
            "row_id":               row_id,
            "source_text":          text,
            "term_count":           len(terms_json),
            "extracted_terms_json": json.dumps(terms_json, ensure_ascii=False),
        })

final_df = pd.DataFrame(final_rows)

print(f"Final output: {len(final_df)} source rows with {len(normalized_df)} total extracted terms.")
display(final_df[["row_id", "term_count", "source_text"]].head())

# Show full structured output for the first row
if len(final_df) > 0:
    print("\nFull structured output for first row:")
    _first_terms = json.loads(final_df.iloc[0]["extracted_terms_json"])
    print(json.dumps(_first_terms, indent=2, ensure_ascii=False))

## Review the Results Before Exporting

Before exporting, ask:

- Do the extracted terms look clinically reasonable?
- Does the best match CURIE seem correct for each term?
- Did the winning model (BioBERT vs SapBERT) make sense?
- Are the `biobert_matched_string` values showing label or synonym matches? (A synonym match can indicate a less precise preferred label.)
- Are any best match cosine scores unexpectedly low (below 0.7)? Those rows may need human review.
- Are there terms where no NR candidates were returned? Those will have `best_match_curie = None`.

If results need improvement, adjust the LLM extraction instructions or review the definitions file, then rerun.

In [ ]:
# ============================================================
# Review Flags
# ============================================================

LOW_COSINE_THRESHOLD = 0.70  # terms with best cosine below this get flagged

if not normalized_df.empty:
    # Flag if LLM review returned a quality that warrants human inspection
    llm_flag = (
        normalized_df["llm_review_quality"].isin(["none", "partial", "parse_failed", "review_failed"])
        if "llm_review_quality" in normalized_df.columns
        else pd.Series(False, index=normalized_df.index)
    )

    normalized_df["needs_human_review"] = (
        normalized_df["best_match_curie"].isna()
        | (normalized_df["extraction_confidence"] == "Low")
        | (
            normalized_df[["best_match_biobert_cosine", "best_match_sapbert_cosine"]]
            .max(axis=1)
            .lt(LOW_COSINE_THRESHOLD)
        )
        | llm_flag
    )

    print(f"Terms needing human review: {normalized_df['needs_human_review'].sum()} / {len(normalized_df)}")

    review_subset = normalized_df[normalized_df["needs_human_review"]][
        ["row_id", "term", "best_match_curie", "best_match_label",
         "best_match_biobert_cosine", "best_match_sapbert_cosine",
         "llm_review_quality", "llm_review_curie",
         "extraction_confidence"]
    ]

    if len(review_subset) > 0:
        print("\nRows flagged for human review:")
        display(review_subset)
    else:
        print("No rows flagged for human review.")

    print("\nWinning model distribution:")
    display(normalized_df["winning_model"].value_counts(dropna=False).reset_index())

    if "llm_review_quality" in normalized_df.columns:
        print("\nLLM review quality distribution:")
        display(normalized_df["llm_review_quality"].value_counts(dropna=False).reset_index())

## Export Results

The export creates an Excel workbook with three sheets:

1. `term_level` — one row per extracted term with flat columns for easy filtering and review.
2. `source_level` — one row per source record with a `extracted_terms_json` column containing the full structured output.
3. `run_metadata` — model name, API settings, row counts, and timestamp.

In [ ]:
# ============================================================
# Export to Excel
# ============================================================

source_stem      = BYOD_DATA_PATH.stem
export_filename  = f"{source_stem}_concept_extraction_{RUN_TIMESTAMP}.xlsx"
export_path      = OUTPUT_DIR / export_filename

# Term-level columns
TERM_COLS = [
    "row_id",
    "term",
    "category",
    "evidence_quote",
    "extraction_confidence",
    "best_match_label",
    "best_match_curie",
    "winning_model",
    "best_match_biobert_cosine",
    "best_match_sapbert_cosine",
    "biobert_matched_string",
    "llm_review_quality",
    "llm_review_curie",
    "llm_review_label",
    "llm_review_reasoning",
    "llm_suggested_query",
    "needs_human_review",
    "other_candidates_json",
]

SOURCE_COLS = ["row_id", "source_text", "term_count", "extracted_terms_json"]

available_term_cols   = [c for c in TERM_COLS   if c in normalized_df.columns]
available_source_cols = [c for c in SOURCE_COLS if c in final_df.columns]

run_metadata_df = pd.DataFrame([
    {"metadata_field": "run_timestamp",           "metadata_value": RUN_TIMESTAMP},
    {"metadata_field": "llm_model",               "metadata_value": MODEL_NAME},
    {"metadata_field": "biobert_model",           "metadata_value": BIOBERT_MODEL_NAME},
    {"metadata_field": "sapbert_url",             "metadata_value": SAPBERT_URL},
    {"metadata_field": "nr_api_url",              "metadata_value": NR_API_URL},
    {"metadata_field": "nr_prefixes",             "metadata_value": NR_PREFIXES},
    {"metadata_field": "nr_limit",                "metadata_value": NR_LIMIT},
    {"metadata_field": "data_path",               "metadata_value": str(BYOD_DATA_PATH)},
    {"metadata_field": "text_column",             "metadata_value": TEXT_COLUMN},
    {"metadata_field": "max_rows_to_run",          "metadata_value": MAX_ROWS_TO_RUN},
    {"metadata_field": "source_rows_processed",   "metadata_value": len(final_df)},
    {"metadata_field": "total_terms_extracted",   "metadata_value": len(normalized_df)},
    {"metadata_field": "terms_flagged_for_review",
     "metadata_value": int(normalized_df["needs_human_review"].sum()) if "needs_human_review" in normalized_df.columns else "N/A"},
    {"metadata_field": "low_cosine_threshold",    "metadata_value": LOW_COSINE_THRESHOLD},
    {"metadata_field": "llm_review_threshold",    "metadata_value": LLM_REVIEW_THRESHOLD},
    {"metadata_field": "allowed_categories",      "metadata_value": ", ".join(ALLOWED_CATEGORIES)},
])

with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
    normalized_df[available_term_cols].to_excel(
        writer, sheet_name="term_level", index=False
    )
    final_df[available_source_cols].to_excel(
        writer, sheet_name="source_level", index=False
    )
    run_metadata_df.to_excel(
        writer, sheet_name="run_metadata", index=False
    )

print("Export complete.")
print(f"Input : {BYOD_DATA_PATH}")
print(f"Output: {export_path.resolve()}")
print("\nSheets:")
print("  - term_level    (one row per extracted term, incl. LLM review columns)")
print("  - source_level  (one row per source record, full JSON output)")
print("  - run_metadata  (model, API settings, counts)")

## What Just Happened

You ran a two-step spec-driven pipeline:

**Step 1 — LLM Extraction**
Each clinical text row was sent to an LLM with a human-defined task specification. The model returned a structured list of phenotype and disease terms, each with a category, an evidence quote, and a confidence level.

**Step 2 — CURIE Normalization**
Each extracted term was sent to the SRI Name Resolution API, which returned up to 25 candidate CURIEs with labels, synonyms, and relevance scores. Candidates were then scored by:
- **BioBERT**: cosine similarity between the original term embedding and the max of (label embedding, synonym embeddings)
- **SapBERT**: embedding-based scores from the RENCI SapBERT API, looked up by CURIE

The best match was the candidate with the highest cosine across both models. All considered candidates are included in the output for transparency and human review.

## Recommended Next Steps

- **Inspect low-confidence and flagged terms** in the `term_level` sheet before using results downstream.
- **Refine the extraction definitions** (`docs/definitions_biomedical-concept-extraction.md`) if the LLM is extracting too broadly or too narrowly.
- **Adjust `NR_PREFIXES`** in the setup cell to restrict or expand the ontologies searched (e.g., add `CHEBI` for chemical entities).
- **Scale up** by increasing `MAX_ROWS_TO_RUN` in the setup cell once the prompt and normalization look correct on the prototype set.
- **Add a second LLM call** following the chained pattern from the CodeBurst notebook — for example, a second pass that reviews the best-match CURIE and rates how well it fits the clinical context.